In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local nocodb")
else:
    print("using aws nocodb")

using aws nocodb


In [3]:
from birddog.database import Database
from birddog.runtime import Runtime
from birddog.database_updater import DatabaseUpdater
from birddog.wiki import (
    get_root_label,
    page_label,
    sequential_page_label,
    )

2026-07-05 17:57:46,253 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com
2026-07-05 17:57:46,262 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-07-05 17:57:46,409 [INFO] Translation is enabled. Using GCP translator
2026-07-05 17:57:46,410 [INFO] Using Google Cloud translation API
2026-07-05 17:57:46,410 [INFO] GoogleCloudTranslator using REST API


In [4]:
runtime = Runtime()
updater = DatabaseUpdater(runtime)

2026-07-05 17:57:46,932 [INFO] PageUpdateManager.init(): detect_environment==local
2026-07-05 17:57:47,405 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-07-05 17:57:47,608 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     4.96    39.00       0.00           24
2026-07-05 17:57:47,811 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-07-05 17:57:48,378 [INFO] WikiDocTracker: base=https://commons.wikimedia.org, namespace=File (id=6)
2026-07-05 17:57:48,584 [INFO] WikiDocTracker: base=https://uk.wikisource.org, namespace=Файл (id=6)
2026-07-05 17:57:48,586 [INFO] KillSwitch: loading thresholds from resou

In [5]:
while updater.refresh_doc_lookups(limit=1000):
    print("finished refresh pass")

2026-07-05 17:58:47,675 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   22.00     3.00    38.77       0.00           24
  uk.wikisource.org:api                  4.00     0.03     4.00       0.00            4
2026-07-05 17:59:47,783 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   28.00    10.00    38.08       0.00           24
  uk.wikisource.org:api                  4.00     0.00     4.00       0.00            4
2026-07-05 18:00:16,812 [INFO] refresh_doc_lookups: writing 1000 updates
2026-07-05 18:00:16,818 [INFO] creating Reserver(table_name=Documents)
finished refresh pass
2026-07-

KeyboardInterrupt: 

2026-07-05 18:03:48,381 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   52.00     9.51    39.00       0.00           24
  uk.wikisource.org:api                  4.00     0.00     4.00       0.00            4


In [ ]:
db = updater._db
#db = Database()

In [ ]:
def record_update(rec):
    label = page_label(rec["title"])
    return {
        "url": rec["url"], 
        "label": label,
        "root_label": get_root_label(label),
        "seq_label": sequential_page_label(label),
    }
    
def do_pass(limit=500):
    rec, _ = db.scan(
        "Pages", 
        view_name="BD:Blank Root Label",
        fields=["url", "title"],
        limit=limit)
    if not rec:
        return False
    rec = [ record_update(r) for r in rec]
    print(f"writing {len(rec)} records")
    db.write("Pages", rec)
    return True

In [ ]:
do_pass(limit=1000)

In [ ]:
while do_pass(limit=1000):
    pass

In [ ]:
def get_owner_ids(db, doc_ids):
    docs = db.read(
        "Documents", 
        doc_ids, 
        fields=["url", "owning_pages"]
    )
    result = {}
    for rec in docs:
        if rec:
            owning_pages = rec.get("owning_pages", [])
            if isinstance(owning_pages, dict):
                owning_pages = [ owning_pages ]
                owning_page_ids = [r["Id"] for r in owning_pages]
                result[rec["Id"]] = owning_page_ids
            else:
                result[rec["Id"]] = []
    return result

In [ ]:
def get_field_lookups(doc_map, page_map, owner_map, field_name):
    result = {}
    for did, rec in doc_map.items():
        result[did] = [page_map[p][field_name] for p in owner_map[did]]
    return result

In [ ]:
def reduce_update_value(field_name, update_value):
    return update_value[0] if update_value else None

In [ ]:
def set_doc_lookup_fields(db, doc_map, page_map, owner_map, lookup_fields, lookup_field_mapping):
    doc_updates = { 
        did: { 
            "url": doc_rec["url"],
            "lookup_status": "valid",
        } for did, doc_rec in doc_map.items() }
    for field_name in lookup_fields:
        updates = get_field_lookups(doc_map, page_map, owner_map, field_name)
        mapped_field_name = lookup_field_mapping.get(field_name, field_name)
        #print(field_name, "-->", mapped_field_name)
        for did, update_value in updates.items():
            doc_updates[did][mapped_field_name] = reduce_update_value(field_name, update_value)
    return list(doc_updates.values())      

In [ ]:
def refresh_lookups_pass(db, limit=100):
    doc_recs, _ = db.scan(
        "Documents", 
        view_name="BD:Need Page Lookups",
        fields=["url", "owning_pages"],
        limit=limit,
    )
    if not doc_recs:
        return False

    _lookup_fields = [
        "label",
        "seq_label",
        "root_label",
        "level",
        "description",
        "native_description",
    ]
    _lookup_field_mapping = {
        "description": "page_description",
        "native_description": "page_native_description",
    }

    owner_map = get_owner_ids(db, [r["Id"] for r in doc_recs])

    page_ids = []
    for p in owner_map.values():
        page_ids.extend(p)
    page_ids = list(set(page_ids))

    page_recs = db.read("Pages", page_ids, fields=_lookup_fields)
    page_map = { r["Id"]: r for r in page_recs }
    doc_map = { r["Id"]: r for r in doc_recs }

    updates = set_doc_lookup_fields(
        db, 
        doc_map, 
        page_map, 
        owner_map, 
        _lookup_fields, 
        _lookup_field_mapping)

    print(f"writing {len(updates)} updates")
    return bool(db.write("Documents", updates))

In [ ]:
while refresh_lookups_pass(db, limit=500):
    pass

In [ ]:
doc_recs, _ = db.scan(
    "Documents", 
    view_name="BD:Need Page Lookups",
    fields="url",
    limit=10,
)


In [ ]:
doc_ids = [r["Id"] for r in doc_recs]
doc_ids

In [ ]:
for did in doc_ids:
    owners = db.get_links("Documents", "owning_pages", did)
    print(did, owners)